# code-switching 발화 교정 번역

In [1]:
#!pip install ipywidgets
#!pip install -U openai-whisper
#!pip install pyaudio
#!pip install sentencepiece

In [2]:
import os
import pyaudio
import numpy as np
import threading
import time
import ipywidgets as widgets
from IPython.display import display
from peft import PeftModel
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from transformers import MarianMTModel, MarianTokenizer
import re

#### 모델 로드

In [3]:
##### 음성인식 모델 로드 
# LoRA 가중치와 기본 모델 경로 설정
base_model_path = "openai/whisper-base"  
lora_model_path = "lora_whisper_model"

# Whisper 기본 모델과 LoRA 병합
base_model = WhisperForConditionalGeneration.from_pretrained(base_model_path)
asr_model = PeftModel.from_pretrained(base_model, lora_model_path)

# Processor 로드
asr_processor = WhisperProcessor.from_pretrained(base_model_path)

In [4]:
##### 번역 모델 로드
model_name = "Helsinki-NLP/opus-mt-ko-en"
trans_tokenizer = MarianTokenizer.from_pretrained(model_name)
trans_model = MarianMTModel.from_pretrained(model_name)

# 한국어-영어 코드스위칭 번역 함수
def translate_korean_phrases(text):
    # 영어 기준으로 분리
    phrases = re.split(r'([a-zA-Z0-9]+)', text)
    translated_phrases = []

    for phrase in phrases:
        # 한국어와 영어를 기준으로 구분
        if re.search(r'[가-힣]', phrase):  # 한국어가 포함된 구
            # 번역 수행
            inputs = trans_tokenizer(phrase, return_tensors="pt", padding=True, truncation=True)
            outputs = trans_model.generate(**inputs)
            translated_phrase = trans_tokenizer.decode(outputs[0], skip_special_tokens=True)
            translated_phrases.append(translated_phrase)
        else:  # 영어 구나 기타 문장은 그대로 유지
            translated_phrases.append(phrase)

    # 공백만 있는 요소 제거
    translated_phrases = [phrase for phrase in translated_phrases if phrase.strip()]

    # 번역된 구문 결합
    return ' '.join(translated_phrases)

/Users/rynn/anaconda3/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


#### 녹음

In [12]:
##### 음성 녹음
# 녹음 설정
FORMAT = pyaudio.paInt16  # 16비트 정수
CHANNELS = 1  # 모노
RATE = 16000  # Whisper 요구 샘플링 레이트
CHUNK = 1024  # 청크 크기

# PyAudio 인스턴스 생성
audio = pyaudio.PyAudio()

# 녹음 데이터 저장 변수
frames = []
is_recording = True

# 진행 표시바 설정
progress_bar = widgets.IntProgress(min=0, max=100, description='녹음 중...')
progress_label = widgets.Label(value="0.00초 경과")
display(progress_bar, progress_label)

def record_audio(duration):
    global frames, is_recording
    stream = audio.open(format=FORMAT, channels=CHANNELS,
                        rate=RATE, input=True,
                        frames_per_buffer=CHUNK)
    
    start_time = time.time()
    while is_recording and (time.time() - start_time < duration):
        data = stream.read(CHUNK)
        frames.append(data)

        elapsed_time = time.time() - start_time
        progress = min(100, int((elapsed_time / duration) * 100))
        progress_bar.value = progress
        progress_label.value = f"{elapsed_time:.2f} 초 경과"
    
    stream.stop_stream()
    stream.close()

def stop_recording():
    global is_recording
    input("\n종료하려면 'Enter'를 누르세요.")
    is_recording = False
    print("\n녹음이 종료되었습니다.")

# 최대 녹음 시간 (초)
max_duration = 30  # Whisper는 30초 제한
record_thread = threading.Thread(target=record_audio, args=(max_duration,))
record_thread.start()

stop_thread = threading.Thread(target=stop_recording)
stop_thread.start()

record_thread.join()
stop_thread.join()

# NumPy 배열로 변환
audio_data = np.frombuffer(b''.join(frames), dtype=np.int16)

IntProgress(value=0, description='녹음 중...')

Label(value='0.00초 경과')


종료하려면 'Enter'를 누르세요. 



녹음이 종료되었습니다.


#### 전사

In [13]:
##### 녹음 데이터 전사
# 16비트 정수형 데이터를 부동소수점 형식으로 변환
print("Whisper 모델로 전사 시작")
audio_data_float = audio_data.astype(np.float32) / 32768.0 

# whisper 입력 데이터 전처리
input_features = asr_processor(audio_data_float, sampling_rate=RATE, return_tensors="pt").input_features

# 모델로 추론
predicted_ids = asr_model.generate(input_features)
transcription = asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

# 전사 결과 출력
print("전사 결과:", transcription)

# PyAudio 종료
audio.terminate()

Whisper 모델로 전사 시작
전사 결과:  그래서 첫번째로는 중요한 변수를 계산하고 그 다음에 그 중요한 변수를 계산한 것을 통해서 인스탄스를 선택하는 방식으로 작동을 하게 됩니다.


#### 오디오 녹음

In [3]:
# 샘플링 레이트 설정
fs = 44100 
audio_data = []
is_recording = True

# 진행 표시바 설정
progress_bar = widgets.IntProgress(min=0, max=100, description='녹음 중...')
progress_label = widgets.Label(value="0.00초 경과")
display(progress_bar, progress_label)

def record_audio(duration):
    global audio_data
    start_time = time.time()  # 시작 시간 기록
    while is_recording and (time.time() - start_time < duration):
        data = sd.rec(int(1 * fs), samplerate=fs, channels=1, dtype='float64')
        sd.wait()
        audio_data.append(data)
        
        elapsed_time = time.time() - start_time
        progress = min(100, int((elapsed_time / duration) * 100))
        progress_bar.value = progress
        progress_label.value = f"{elapsed_time:.2f} 초 경과"

def stop_recording():
    global is_recording
    input("\n종료하려면 'Enter'를 누르세요.")
    is_recording = False
    print("\n녹음이 종료되었습니다.")

max_duration = 60

# 스레드 시작
record_thread = threading.Thread(target=record_audio, args=(max_duration,))
record_thread.start()

stop_thread = threading.Thread(target=stop_recording)
stop_thread.start()

# 스레드가 종료될 때까지 대기
record_thread.join()
stop_thread.join()

IntProgress(value=0, description='녹음 중...')

Label(value='0.00초 경과')


종료하려면 'Enter'를 누르세요. 



녹음이 종료되었습니다.


#### 녹음 데이터 재생

In [5]:
# 녹음된 데이터 재생
print("녹음된 데이터를 재생합니다.")

# 녹음된 데이터 NumPy 배열로 변환
audio_data = np.frombuffer(b''.join(frames), dtype=np.int16)

# 오디오 데이터 재생
sd.play(audio_data, RATE)
sd.wait()
print("재생이 완료되었습니다.")

녹음된 데이터를 재생합니다.


NameError: name 'frames' is not defined

#### 번역

In [14]:
##### 번역
translated_text = translate_korean_phrases(transcription)
print("Translated Text:", translated_text)

Translated Text: So the first thing you do is you calculate an important variable, and then you calculate that important variable, and it works in a way that you choose instans.
